
# Transform Payments Data 

1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment_time
2. Map payment_status to contain descriptive values 
  (1- Success, 2 - Pending, 3- Cancelled, 4- Failed)
3. Write transformed data to the Silver Schema

In [0]:
SELECT * FROM gizmobox.bronze.ext_payments;


## 1. Extract Date and Time from payment_timestamp and create new columns payment_date and payment time

In [0]:
SELECT *, 
    CAST(DATE_FORMAT(payment_timestamp, 'yyyy-MM-dd') AS DATE) as payment_date, 
    DATE_FORMAT(payment_timestamp, 'HH:mm:ss') as payment_time
FROM gizmobox.bronze.ext_payments


## 2. Map payment_status to contain descriptive values (1- Success, 2 - Pending, 3- Cancelled, 4- Failed)

In [0]:
SELECT payment_id, 
      order_id, 
    CAST(DATE_FORMAT(payment_timestamp, 'yyyy-MM-dd') AS DATE) as payment_date, 
    DATE_FORMAT(payment_timestamp, 'HH:mm:ss') as payment_time, 
    (CASE 
        WHEN payment_status = 1 THEN 'success'
        WHEN payment_status = 2 THEN 'pending'
        WHEN payment_status = 3 THEN 'cancelled'
        WHEN payment_status = 4 THEN 'failed'
    ELSE 'unknown'
    END) AS payment_status, 
    payment_method
FROM gizmobox.bronze.ext_payments


## 3. Write transformed data into Silver Schema 

In [0]:
CREATE TABLE IF NOT EXISTS gizmobox.silver.payments
AS 
SELECT payment_id, 
      order_id, 
    CAST(DATE_FORMAT(payment_timestamp, 'yyyy-MM-dd') AS DATE) as payment_date, 
    DATE_FORMAT(payment_timestamp, 'HH:mm:ss') as payment_time, 
    (CASE 
        WHEN payment_status = 1 THEN 'success'
        WHEN payment_status = 2 THEN 'pending'
        WHEN payment_status = 3 THEN 'cancelled'
        WHEN payment_status = 4 THEN 'failed'
    ELSE 'unknown'
    END) AS payment_status, 
    payment_method
FROM gizmobox.bronze.ext_payments

In [0]:
-- successfully created payments table in the silver schema
SELECT * FROM gizmobox.silver.payments;

In [0]:
DESCRIBE EXTENDED gizmobox.silver.payments;

-- Check for the type MANAGED, although the data sits in the s3 location if can be managed by the databricks, basically means we can delete or drop the table so the metadata as well as the data will be deleted.